# FibrosisLit vs. Elicit / Consensus / Semantic Scholar — Comparison Analysis

Loads the latest FibrosisLit benchmark run and the manually-filled comparison CSV,
computes per-tool per-tier accuracy, and displays a head-to-head summary table.

**Prerequisites:**
1. FibrosisLit benchmark run exists in `benchmarks/results/results_*.csv`
2. `benchmarks/comparison/comparison_template.csv` has been filled in for at least one tool

**Verdict correctness rules:**
- WELL_SUPPORTED: correct = `SUPPORTED`
- CONTESTED: correct = `CONTESTED`
- OVERCLAIMED: correct = `UNSUPPORTED` **or** `INSUFFICIENT_EVIDENCE`

In [ ]:
import sys, pathlib, glob
_root = pathlib.Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print('Project root:', _root)

In [ ]:
import pandas as pd

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.1%}'.format)

## Load FibrosisLit results (latest benchmark run)

In [ ]:
results_files = sorted(glob.glob(str(_root / 'benchmarks/results/results_*.csv')))
if not results_files:
    raise FileNotFoundError('No benchmark results found. Run benchmark_runner.py first.')

fl_csv = results_files[-1]
print(f'Using: {fl_csv}')

fl_df = pd.read_csv(fl_csv)
# 'verdict' is the final two-vote verdict; 'correct' is the boolean correctness column
fl_df = fl_df[['claim_id', 'tier', 'expected_verdict', 'verdict', 'verdict_confidence', 'correct']].copy()
fl_df.rename(columns={'verdict': 'fibrosislit_verdict', 'correct': 'fibrosislit_correct'}, inplace=True)
fl_df['fibrosislit_correct'] = fl_df['fibrosislit_correct'].astype(bool)
print(f'{len(fl_df)} rows loaded.')
fl_df.head(3)

## Load comparison CSV (manually filled)

In [ ]:
comp_path = _root / 'benchmarks/comparison/comparison_template.csv'
comp_df = pd.read_csv(comp_path)
print(f'Comparison template: {len(comp_df)} rows, {comp_df.shape[1]} columns')

# Count how many rows have been filled per tool
for tool in ['elicit', 'consensus', 'semantic_scholar']:
    filled = comp_df[f'{tool}_verdict'].notna() & (comp_df[f'{tool}_verdict'] != '')
    print(f'  {tool}: {filled.sum()}/22 rows filled')

## Merge and compute correctness

In [ ]:
OVERCLAIMED_CORRECT = {'UNSUPPORTED', 'INSUFFICIENT_EVIDENCE'}

def is_correct(actual: str, expected: str) -> bool:
    """Correctness rule: UNSUPPORTED and INSUFFICIENT_EVIDENCE are both valid for OC claims."""
    if not isinstance(actual, str) or actual.strip() == '':
        return False
    if expected in OVERCLAIMED_CORRECT:
        return actual.strip() in OVERCLAIMED_CORRECT
    return actual.strip() == expected.strip()


# Merge on claim_id
df = fl_df.merge(
    comp_df[[
        'claim_id', 'claim', 'failure_mode', 'scoring_notes',
        'elicit_verdict',           'elicit_response',
        'consensus_verdict',        'consensus_response',
        'semantic_scholar_verdict', 'semantic_scholar_response',
    ]],
    on='claim_id',
)

# Compute correctness for external tools
for tool in ['elicit', 'consensus', 'semantic_scholar']:
    df[f'{tool}_correct'] = df.apply(
        lambda r: is_correct(r[f'{tool}_verdict'], r['expected_verdict']),
        axis=1,
    )

print(f'Merged dataframe: {len(df)} rows')

## Overall accuracy — all 22 claims

In [ ]:
tools = ['fibrosislit', 'elicit', 'consensus', 'semantic_scholar']

overall_rows = []
for tool in tools:
    n_filled = df[f'{tool}_correct'].notna().sum() if tool != 'fibrosislit' else len(df)
    n_correct = df[f'{tool}_correct'].sum()
    overall_rows.append({
        'Tool': tool.replace('_', ' ').title(),
        'Correct': int(n_correct),
        'Total':   int(n_filled),
        'Accuracy': n_correct / n_filled if n_filled > 0 else float('nan'),
    })

overall_df = pd.DataFrame(overall_rows)
display(overall_df.style.format({'Accuracy': '{:.1%}'}).hide(axis='index'))

## Per-tier accuracy

In [ ]:
tier_rows = []
for tier in ['well_supported', 'contested', 'overclaimed']:
    g = df[df['tier'] == tier]
    row = {'Tier': tier, 'N': len(g)}
    for tool in tools:
        row[tool.replace('_', ' ').title()] = g[f'{tool}_correct'].mean()
    tier_rows.append(row)

tier_df = pd.DataFrame(tier_rows)
tool_cols = [t.replace('_', ' ').title() for t in tools]
display(
    tier_df.style
    .format({c: '{:.1%}' for c in tool_cols})
    .hide(axis='index')
)

## Full verdict table

In [ ]:
display_cols = [
    'claim_id', 'tier', 'expected_verdict',
    'fibrosislit_verdict', 'elicit_verdict', 'consensus_verdict', 'semantic_scholar_verdict',
]

def mark_correct(val, expected, tool):
    if not isinstance(val, str) or val.strip() == '':
        return val
    ok = is_correct(val, expected)
    return f"{val} {'✓' if ok else '✗'}"

styled = df[display_cols + ['expected_verdict']].copy()
for tool in tools:
    col = f'{tool}_verdict'
    if col in styled.columns:
        styled[col] = styled.apply(
            lambda r: mark_correct(r[col], r['expected_verdict'], tool), axis=1
        )

display(
    styled[display_cols]
    .style.set_properties(**{'text-align': 'left'})
    .hide(axis='index')
)

## Failure mode analysis — where each tool fails

For each tool, shows claims it got wrong and the expected failure mode.

In [ ]:
for tool in tools:
    failed = df[~df[f'{tool}_correct'].astype(bool)]
    tool_label = tool.replace('_', ' ').title()
    print(f'\n=== {tool_label}: {len(failed)} failures ===')
    if failed.empty:
        print('  None.')
        continue
    for _, row in failed.iterrows():
        actual = row.get(f'{tool}_verdict', 'N/A')
        print(f"  {row['claim_id']} [{row['tier']}]")
        print(f"    Expected: {row['expected_verdict']}  Got: {actual}")
        print(f"    Failure mode: {row['failure_mode'][:120]}")